## Step 1. Imports and setup

In [ ]:
# %pip install requests
import pandas as pd
import requests

In [ ]:
df = pd.read_csv("data.csv")
# store your dataset in the same directory/folder as this script
# replace with your file name
# ensure that your csv file has columns titled Address and Town

## Step 2. Creating a new csv

Create a csv file with just columns to hold the values for
Unique ID, Street address, City, State, ZIP
Use the specific Address, City column from our csv file
create a Unique ID column using index +1
Drop the header
Have the Unique ID and Street Address values in this csv, separated by newline characters for the rows

In [ ]:
df = df.reset_index(drop=True)
df["Unique ID"] = range(1, len(df) + 1)
df_address = pd.DataFrame({
    "Unique ID": df["Unique ID"],
    "Street Address": df["Address"],
    "City": df["Town"],
    # State filled as MA for now
    "State": ["MA"] * len(df),
    "ZIP": [None] * len(df)
})

df_address.to_csv("addresses.csv", index=False, header=False)

## Step 3: Batch Addresses - standardizing addresses

Request a response from the api using a batch request
Handle the response appropriately 
Receive the response
In our dataframe 
Add entires into a 3rd column based on the Matched/Unmatched section
Add entires into a 4th column based on what kind of match 
Exact
Tie
Add the matched address received into a 5th column 

In [ ]:
import csv
from io import StringIO

API_URL = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
params = {
    'benchmark': 'Public_AR_Current',
    'vintage': 'Current_Current',
}
with open('addresses.csv', 'rb') as file:
    files = {'addressFile': file}
    response = requests.post(API_URL, data=params, files=files)

    if response.status_code == 200:
        print("Geocoding successful. Generating results...")
        results = csv.reader(StringIO(response.text))
        response_rows = list(results)  # convert to list so we can iterate multiple times
        # Build dict keyed by UID (int)
        response_dict = {int(row[0]): row for row in response_rows}
        # Prepare lists for new columns
        match_statuses = []
        match_types = []
        matched_addresses = []

        # Assuming df_address has rows matching UID 1..N
        for uid in range(1, len(df_address) + 1):
            resp = response_dict.get(uid)
            if resp and len(resp) > 2:
                match_statuses.append(resp[2])  # Match Status
                match_types.append(resp[3] if len(resp) > 3 else None)  # Match Type
                matched_addresses.append(resp[4] if len(resp) > 4 else None)  # Matched Address
            else:
                match_statuses.append(None)
                match_types.append(None)
                matched_addresses.append(None)

        # Add these new columns to the dataframe
        df_address["Match Status"] = match_statuses
        df_address["Match Type"] = match_types
        df_address["Matched Address"] = matched_addresses
        df_address.to_csv("matched_addresses.csv", index=False)
    else:
        print("Request failed:", response.status_code)


In [ ]:
df_address.groupby(['Match Status']).describe()

In [ ]:
df_address.groupby(['Match Status', 'Match Type']).describe()

## Step 4: Accessing the Corresponding MSAs one row at a time

Now, for every row in our updated dataframe
Send a request to the api using this matched address
Handle appropriately
Store the Metropolitan Statistical or Combined Statistical Area (new census category - msa is outdated?) in a column 
Leave null otherwise

In [ ]:
df_address.describe(include="all")

In [ ]:
def getMSA(matched_address):
    params = {
        'address': matched_address,
        'benchmark': 'Public_AR_Current',
        'vintage': 'Current_Current',
        'layers': '92,93',
        'format': 'json'
    }
    if (matched_address == None):
        print("No Matched Address from previous call")
        return
    response = requests.get('https://geocoding.geo.census.gov/geocoder/geographies/onelineaddress', params=params)
    if response.status_code == 200:
        data = response.json()
        matches = data.get('result', {}).get('addressMatches', [])
        if matches:
            geographies = matches[0].get('geographies', {})
            msa_info = geographies.get('Metropolitan Statistical Areas', [])
            if msa_info:
                return(msa_info[0]['BASENAME'])
            else:
                return None
        else:
            return None
    else:
        print("Request failed:", response.status_code)
        return None

In [ ]:
# Example:
i = 105
print(df_address.iloc[i]['Matched Address'])
test_address = df_address.iloc[i]['Matched Address'];
print(getMSA(test_address))

## Step 5: Creating a new csv from your dataframe 

In [ ]:
df_address["MSA"] = None  # Ensure the column exists

for i, row in df_address.iterrows():
    matched_address = row.get("Matched Address")
    uid = row["Unique ID"]

    if pd.notnull(matched_address) and isinstance(matched_address, str) and matched_address.strip():
        msa_name = getMSA(matched_address)
        df_address.loc[df_address["Unique ID"] == uid, "MSA"] = msa_name
    else:
        print(f"Skipping UID {uid} — no valid matched address.")

In [ ]:
df_address.MSA

In [ ]:
df_address.to_csv("adresses_MSA.csv")

## Step 6: Merging the new csv with your original csv

In [ ]:
df_merged = pd.merge(df, df_address, on="Unique ID", how="left")
df_merged.to_csv("with_geocoding.csv")